# R6 D6: planner data for the 15 checkpoints (GPU runtime)

Runs `experiments/r6_tdmpc2/planner_collect.py` under deviation D6 (tag
`prereg-r6-d3`), sections (a), (b) and (e):

1. the 15 checkpoints from the pinned Hugging Face revision, refused unless each
   SHA-256 matches D6's table (read from the tag);
2. the regenerated D4 observations from `MyDrive/layernorm-lens-r6/data/r6/`, used
   only if `meta_d4_regeneration.json` records `match: true` and each file matches
   `d4_obs_manifest.csv`;
3. the positive and negative controls, then the encoder agreement gate for both
   pre-release checkpoints. A control that does not behave as stated halts
   everything (exit status 4). A failed gate halts only the pre-release runs (exit
   status 5 at the end);
4. the 13 public-layout checkpoints, then the 2 pre-release ones, each with 50 episodes
   and `eval_mode=True`.

No criterion quantity (Inside, Populated, Sharp), no G1, no consistency (c) and no
susceptibility (g) is computed.

**Runtime > Change runtime type > T4 GPU.** Run the **smoke** cell first (controls,
gate and 1 episode of cartpole-swingup seed 1) and report its output before the full
run. The smoke outputs are outcome data: they are kept on Drive and listed, but never
used for any rule.

The full run is resumable. After a disconnect, rerun cells 1–4, then cell 7:
checkpoints already complete on Drive are skipped.

In [ ]:
import collections, os, subprocess, sys
def sh(cmd, ok=(0,), tail=40):
    """Run a shell command, streaming its output into the cell line by line. On an
    exit status not in `ok`, print the last `tail` lines again and stop the cell."""
    print("$", cmd, flush=True)
    last = collections.deque(maxlen=tail)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        last.append(line)
    p.wait()
    if p.returncode not in ok:
        print(f"\n--- exit status {p.returncode}; last {len(last)} lines ---")
        sys.stdout.write("".join(last))
        raise RuntimeError(f"exit status {p.returncode}: {cmd}")
    return p.returncode

# 1. Repository (this session's branch) with its tags. GITHUB_TOKEN Colab secret if private.
import os
BRANCH = "claude/new-session-0r0qe0"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = (f"https://{token}@github.com/binoygeorge97/layernorm-lens" if token
       else "https://github.com/binoygeorge97/layernorm-lens")
if not os.path.isdir("/content/layernorm-lens"):
    sh(f"git clone --branch {BRANCH} {url} /content/layernorm-lens")
os.chdir("/content/layernorm-lens")
sh(f"git fetch -q --tags origin && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH}")
sh("git log --oneline -1")
sh('for t in prereg-r6 prereg-r6-d1 prereg-r6-d2 prereg-r6-d3; do printf "%-14s " $t; '
   'test "$(git cat-file -t $t)" = tag || exit 1; git rev-parse $t^{commit}; done')
sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")

In [ ]:
# 2. Google Drive: the D4 observations are read from, and all outputs copied to,
#    MyDrive/layernorm-lens-r6/.
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/layernorm-lens-r6"
sh(f'ls -la "{DRIVE}/data/r6" | head -20')

In [ ]:
# 3. tdmpc2 at the pinned commit.
sh("test -d checkpoints/tdmpc2_src || git clone -q https://github.com/nicklashansen/tdmpc2 checkpoints/tdmpc2_src")
sh("git -C checkpoints/tdmpc2_src checkout -q e9f59321933cbc8e11a002b842adc7d4ffae8ff1 && git -C checkpoints/tdmpc2_src rev-parse HEAD")

In [ ]:
# 4. tdmpc2's pinned environment (docker/environment.yaml pip section) in a Python 3.11
#    virtualenv, plus pyyaml and pytest. Stops on any install error; the checks and
#    versions print only after a successful install.
import yaml
VENV = "/content/tdmpc2-venv"
PY = f"{VENV}/bin/python"
env = yaml.safe_load(open("checkpoints/tdmpc2_src/docker/environment.yaml"))
pins = [p for d in env["dependencies"] if isinstance(d, dict) for p in d["pip"]]
open("/content/tdmpc2-pins.txt", "w").write("\n".join(pins + ["pyyaml", "pytest"]) + "\n")
print(pins)
sh("pip install -q uv")
if not os.path.exists(PY):
    sh(f"uv venv -q --python 3.11 {VENV}")
sh(f"uv pip install -q --python {PY} -r /content/tdmpc2-pins.txt")
sh(f"MUJOCO_GL=egl {PY} -c \"from dm_control import suite; suite.load('cartpole', 'swingup', task_kwargs={{'random': 0}}); import torch; assert torch.cuda.is_available(); print('dm_control.suite and CUDA ok')\"")
sh(f"{PY} -c \"import platform, importlib.metadata as m; print('python', platform.python_version()); "
   "[print(p, m.version(p)) for p in ('torch','tensordict','torchrl','mujoco','dm_control','numpy','gymnasium')]\"")

In [ ]:
# 5. Pipeline checks in this environment (no checkpoints needed).
sh(f"{PY} -m pytest -q tests/test_planner_collect.py")

In [ ]:
# 6. SMOKE: controls, gate, then 1 episode (environment seed 0) of cartpole-swingup seed 1
#    if the gate passed. Outputs go to Drive under data/r6/planner_smoke/ and
#    results/r6/planner_smoke/. Exit status: 0 ok, 4 a control did not behave as stated
#    (everything halted), 5 the gate failed (no pre-release run). Report the output.
status = sh(f'cd /content/layernorm-lens && MUJOCO_GL=egl {PY} experiments/r6_tdmpc2/planner_collect.py '
            f'--config experiments/r6_tdmpc2/config.yaml --drive "{DRIVE}" --smoke', ok=(0, 4, 5))
print("exit status:", status)

In [ ]:
# 7. FULL RUN (only after the smoke run has been reported). Resumable: complete
#    checkpoints on Drive are skipped. About 10 minutes per checkpoint on a T4.
status = sh(f'cd /content/layernorm-lens && MUJOCO_GL=egl {PY} experiments/r6_tdmpc2/planner_collect.py '
            f'--config experiments/r6_tdmpc2/config.yaml --drive "{DRIVE}"', ok=(0, 4, 5))
print("exit status:", status)

In [ ]:
# 8. Summaries, and the result files to commit (the run printed the exact git add -f
#    command). Observation data stays on Drive.
import pandas as pd
for f in ("planner_returns.csv", "encoder_gate.csv", "planner_obs_manifest.csv"):
    p = f"results/r6/planner/{f}"
    if os.path.exists(p):
        print(f); display(pd.read_csv(p))
dirs = [d for d in ("results/r6/planner", "results/r6/planner_smoke") if os.path.isdir(d)]
sh("zip -q -r /content/r6_planner_results.zip " + " ".join(dirs))
from google.colab import files
files.download("/content/r6_planner_results.zip")